# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane CTR/ENGAGEMENT OPPORTUNITY SCORING as an ML task

My lane is a SCORING task.
The core problem is: pages are appearing in search but not getting engagement, and we need to see how they are actually doing.
So the model scores each page to show how well or how poorly it is performing.

Ranking is not the primary task type, it is the output of scoring.
Once the model produces a score for every page, we can order those scores from worst to best.
That ranked list is what tells a client which pages need to be fixed first.
So ranking is the application of the model's output, not the modeling task itself.

Classification could be used at the very end, as a secondary step.
Once pages are scored, we could split them into two simple categories, such as best scored versus worst scored, just to help decide which group to focus on first.
But that classification step depends entirely on the score already existing.
It cannot happen without scoring first, which is why scoring is the true underlying ML task, and both ranking and classification are things we build on top of it.

In [1]:
# Illustration: CTR/Engagement Opportunity Scoring as a SCORING task
# This is a small conceptual example, not the real dataset (that comes in Section 4)

import pandas as pd

# A model's core output for this lane: a continuous score per page
# Higher score = more opportunity (page appearing in search but underperforming on engagement)
example_scores = pd.DataFrame({
    "page_id": ["page_1", "page_2", "page_3", "page_4"],
    "opportunity_score": [0.82, 0.15, 0.64, 0.41]
})

print("Step 1: Scoring is the actual ML task, the model outputs a continuous number per page")
print(example_scores)

# Ranking is NOT a separate modeling task, it is just sorting the scores
print("\nStep 2: Ranking is the output of scoring, sorting the same scores worst to best")
ranked = example_scores.sort_values("opportunity_score", ascending=False).reset_index(drop=True)
print(ranked)

# Classification could be applied AFTER scoring exists, as an optional secondary step
print("\nStep 3: Classification can be layered on top, only after scores already exist")
ranked["priority_group"] = ranked["opportunity_score"].apply(
    lambda s: "focus_first" if s >= 0.5 else "lower_priority"
)
print(ranked)

Step 1: Scoring is the actual ML task, the model outputs a continuous number per page
  page_id  opportunity_score
0  page_1               0.82
1  page_2               0.15
2  page_3               0.64
3  page_4               0.41

Step 2: Ranking is the output of scoring, sorting the same scores worst to best
  page_id  opportunity_score
0  page_1               0.82
1  page_3               0.64
2  page_4               0.41
3  page_2               0.15

Step 3: Classification can be layered on top, only after scores already exist
  page_id  opportunity_score  priority_group
0  page_1               0.82     focus_first
1  page_3               0.64     focus_first
2  page_4               0.41  lower_priority
3  page_2               0.15  lower_priority


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
### 2. Target or proxy

**What would you predict?**
A continuous engagement score for each page, showing how much a page is struggling to convert its search visibility into real engagement.

**Where does that label come from — observed outcome or a defined rule?**
A defined rule. This is a proxy, not an observed outcome. There is no column that directly records whether a page deserves attention. Instead, the label is built by engineering a composite feature from CTR and engagement rate, normalized to a common scale and combined into one engagement composite score. Pages with very low impressions are excluded, since low visibility pages cannot give a meaningful signal either way. A threshold is then set using the actual distribution of this composite score in the data, flagging the lowest percentile range as declining and the higher percentile range as performing fine. This makes the label a defined rule grounded in the real data, not an observed fact and not an arbitrary guess.

### 2. Target or proxy

**What would you predict?**
A continuous engagement score for each page, showing how much a page is struggling to convert its search visibility into real engagement.

**Where does that label come from — observed outcome or a defined rule?**
A defined rule. This is a proxy, not an observed outcome. There is no column that directly records whether a page deserves attention. Instead, the label is built by engineering a composite feature from CTR and engagement rate, normalized to a common scale and combined into one engagement composite score. Pages with very low impressions are excluded, since low visibility pages cannot give a meaningful signal either way. A threshold is then set using the actual distribution of this composite score in the data, flagging the lowest percentile range as declining and the higher percentile range as performing fine. This makes the label a defined rule grounded in the real data, not an observed fact and not an arbitrary guess.

In [3]:
import pandas as pd, numpy as np

url = "https://raw.githubusercontent.com/KhadijaHussnainMLEngineer/MLPipeline_MachineLearningFlyRankAI/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

print(df["ctr"].describe())
print(df["engagement_rate"].describe())

# Normalize each metric to 0-100 scale using min-max scaling
def normalize(col):
    return (col - col.min()) / (col.max() - col.min()) * 100

df["ctr_norm"] = normalize(df["ctr"])
df["engagement_norm"] = normalize(df["engagement_rate"])

# Composite: average of the two normalized signals
df["engagement_composite_score"] = (df["ctr_norm"] + df["engagement_norm"]) / 2

df[["ctr", "engagement_rate", "ctr_norm", "engagement_norm", "engagement_composite_score"]].head(10)

# Only trust the composite score where visibility is real
df["has_enough_visibility"] = df["impressions_90d"] >= 500  # pages actually worth judging
df["valid_composite_score"] = np.where(df["has_enough_visibility"], df["engagement_composite_score"], np.nan)
df["valid_composite_score"].describe()

# Use percentiles from the actual data to set the threshold
low_cutoff = df["valid_composite_score"].quantile(0.25)
high_cutoff = df["valid_composite_score"].quantile(0.75)

print(f"Declining cutoff (25th percentile): {low_cutoff:.3f}")
print(f"Performing fine cutoff (75th percentile): {high_cutoff:.3f}")

# Build the actual proxy label
df["engagement_proxy_label"] = np.where(
    df["valid_composite_score"] <= low_cutoff, "declining",
    np.where(df["valid_composite_score"] >= high_cutoff, "performing_fine", "middle")
)

# Show how many pages fall into each group
print(df["engagement_proxy_label"].value_counts())

(30000, 44)
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64
count    30000.000000
mean         2.534520
std          8.310096
min          0.000000
25%          0.000000
50%          0.000000
75%          1.350000
max        100.000000
Name: engagement_rate, dtype: float64
Declining cutoff (25th percentile): 0.050
Performing fine cutoff (75th percentile): 2.045
engagement_proxy_label
middle             21442
declining           4371
performing_fine     4187
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### 3. Success metric

The metric I will use is **Precision@K**.

This fits because my task is scoring and ranking, not classification. I only need the model to be right about the pages at the very top of the list, not every page.

To test this honestly, I first checked correlations between raw, independent columns and the two features that built my Section 2 label (ctr and engagement_rate), rather than guessing combinations. The strongest real, non-circular signal was engaged_sessions_90d, correlated at 0.135 with engagement_rate, still weak, but the best available.

Using engaged_sessions_90d alone as a real stand-in prediction, Precision@50 came out to 0.160, compared to a random guess baseline of 0.146. This is only a small improvement, but it is a real, honest signal, not something forced or circular.

"Good" means Precision@K clearly higher than this random baseline. This result shows only a slight edge from one single feature, which supports why a full model considering many features together is actually needed, since no single raw signal on its own predicts decline strongly.

In [15]:
# Check which columns actually correlate with ctr and engagement_rate
numeric_df = df.select_dtypes(include=[np.number])

ctr_correlations = numeric_df.corr()["ctr"].sort_values(ascending=False)
engagement_correlations = numeric_df.corr()["engagement_rate"].sort_values(ascending=False)

print("Top correlations with CTR:")
print(ctr_correlations.head(10))

print("\nTop correlations with engagement_rate:")
print(engagement_correlations.head(10))

Top correlations with CTR:
ctr                           1.000000
ctr_norm                      1.000000
engagement_composite_score    0.442780
valid_composite_score         0.141236
engagement_norm               0.096903
engagement_rate               0.096903
scroll_rate                   0.012955
clicks_90d                    0.010609
clicks_prev_30d               0.010217
content_age_days              0.009460
Name: ctr, dtype: float64

Top correlations with engagement_rate:
engagement_norm               1.000000
engagement_rate               1.000000
valid_composite_score         0.998925
engagement_composite_score    0.935317
scroll_rate                   0.162611
engaged_sessions_90d          0.135066
ctr_norm                      0.096903
ctr                           0.096903
days_with_impressions         0.063950
days_with_sessions            0.044146
Name: engagement_rate, dtype: float64


In [14]:
y_true = (df["engagement_proxy_label"] == "declining").astype(int)

# engaged_sessions_90d is the strongest real, independent correlation found
# Lower engaged_sessions_90d likely means more at risk of declining
stand_in_score_v3 = df["engaged_sessions_90d"]

def precision_at_k_lowest(scores, labels, k):
    order = np.argsort(np.asarray(scores))  # lowest = most likely declining
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

precision_50_v3 = precision_at_k_lowest(stand_in_score_v3.fillna(stand_in_score_v3.max()), y_true, 50)
random_baseline = y_true.mean()

print(f"Precision@50 using engaged_sessions_90d alone: {precision_50_v3:.3f}")
print(f"Random guess baseline: {random_baseline:.3f}")


Precision@50 using engaged_sessions_90d alone: 0.160
Random guess baseline: 0.146


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*



One row equals one page, one distinct piece of content, identified by content_id. This was verified directly: the dataset has 30,000 total rows and exactly 30,000 unique content_id values, confirming no repeats.

Each page belongs to one of 32 clients, identified by client_id, but content_id alone uniquely identifies each row.

Each row describes a single page's search and engagement performance, including columns such as impressions_90d, avg_position, ctr, engagement_rate, content_age_days, and word_count.

In [10]:
# Verify the unit of analysis
print("Total rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())
print("Unique client_id:", df["client_id"].nunique())

# Show the actual dataframe as proof
df[["content_id", "client_id", "impressions_90d", "avg_position", "ctr", "engagement_rate", "content_age_days", "word_count"]].head(10)

Total rows: 30000
Unique content_id: 30000
Unique client_id: 32


,content_id,client_id,impressions_90d,avg_position,ctr,engagement_rate,content_age_days,word_count
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,5.88,187,3221.0
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,0.00,445,2481.0
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,0.00,141,3515.0
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,1.28,463,NaN
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,0.00,263,2803.0
5,content_d4084a4bc775,client_f369cb89fc,3970,8.5,0.03,0.00,147,3080.0
6,content_9a34b442b552,client_8722616204,20,7.0,0.00,0.00,90,3059.0
7,content_a63219c6e95a,client_19581e27de,1724,21.2,0.06,3.57,445,NaN
8,content_5e6c160719bc,client_6208ef0f77,32574,46.0,0.09,5.88,90,3807.0
9,content_c27558df2b0c,client_19581e27de,1240,4.9,0.16,0.00,257,NaN


In [8]:
print("Total rows:", len(df))
print("All columns:", df.columns.tolist())

Total rows: 30000
All columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'ctr_norm', 'engagement_norm', 'engagement_composite_score', 'has_enough_visibility', 'valid_composite_score', 'engagement_proxy_label']


In [9]:
print("Total rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())
print("Unique client_id:", df["client_id"].nunique())

Total rows: 30000
Unique content_id: 30000
Unique client_id: 32


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*



A fixed if-statement often wastes time by flagging pages that look bad on the surface but are actually fine. For example, a rule might flag a page as "needs review" just because one number looks low, when in reality the page is positioned okay once you consider it properly. This happens because a simple rule cannot weigh multiple signals together the way a model can, it just checks one condition at a time.

A fixed rule is also too narrow to handle something like a composite metric. Building a score from multiple normalized features, checking thresholds based on real data distribution, and combining several signals into one number is a calculation a model handles naturally, but an if-statement cannot reasonably scale to. The more signals and interactions involved, the less realistic it becomes to hand-write a rule for it.

This is why ML fits better here, it can weigh several features together, avoid false alarms a simple rule would generate, and handle the kind of layered calculation a composite score requires.

In [7]:
# Small illustration: a simple if-statement rule can create false alarms

# Example: rule says "if CTR is low, flag as needs review" -- checking only ONE signal
example_pages = pd.DataFrame({
    "page_id": ["page_1", "page_2", "page_3"],
    "ctr": [0.05, 0.05, 0.05],          # all three pages have the same low CTR
    "avg_position": [2, 25, 48],         # but very different search positions
    "impressions_90d": [50000, 8000, 300]
})

# Simple rule: only checks CTR, ignores everything else
example_pages["simple_rule_flag"] = example_pages["ctr"] < 0.1

print("Simple rule flags all three pages the same way, even though context is very different:")
print(example_pages)

Simple rule flags all three pages the same way, even though context is very different:
  page_id   ctr  avg_position  impressions_90d  simple_rule_flag
0  page_1  0.05             2            50000              True
1  page_2  0.05            25             8000              True
2  page_3  0.05            48              300              True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.